<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_03_transfer_and_fine_tuning_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 03 — A Second Machine, and Ten Labels Each

**Paired with L6.2 · Post-Training and Reinforcement Learning**

You have a classifier that tells balanced from imbalance from bearing fault on
machine A, trained on three hundred and sixty labelled samples. Maintenance now
wants the same thing on machine B.

Machine B has a different accelerometer, a different gain, and is bolted to a
different part of the frame. The faults are the same physics; the numbers are
not. And you have **ten labelled samples per class**, because labelling means
running a machine until it breaks.

## What you will do

1. Watch the machine-A model fail on machine B, and measure how badly.
2. Try three ways to get a machine-B model, all on the same thirty labels:
   train from scratch, retrain only the last layer, fine-tune everything.
3. Find the learning rate at which fine-tuning destroys what it was given.
4. Ask how many labels you actually needed.

## Why this is here

L11 hands you a ResNet-18 pretrained on ImageNet and asks you to replace its
last layer and fine-tune on a few hundred photographs of a corridor. That is
this notebook with a bigger network and pictures instead of accelerometers. If
the shape of the argument is clear here, L11 is a detail.

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
print("torch", torch.__version__, "| output dir:", core.OUTPUT_DIR)

## 1 · Two machines, same faults

Look at them together before modelling anything. The classes are in the same
*arrangement* on both — that is the structure a transferred model can reuse —
but they sit somewhere else and at a different angle.

In [ ]:
XA, yA = core.vibration_dataset()
XB, yB = core.machine_b_dataset()

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), sharex=True, sharey=True)
core.plot_classes(XA, yA, ax=axes[0], title="machine A — 360 labelled samples")
core.plot_classes(XB, yB, ax=axes[1], title="machine B — 180, mostly unlabelled")
plt.show()

print("machine A feature means:", XA.mean(axis=0).round(3))
print("machine B feature means:", XB.mean(axis=0).round(3))

**What you should see.** Two clouds of three classes with the same internal
arrangement, machine B's shifted right, squashed on the vertical axis and
rotated. This is *covariate shift*: the relationship between fault and
vibration is unchanged, but the measurement of it is not.

## 2 · The machine-A model

Train it properly on all of machine A, then leave it alone. It is the asset you
are trying to reuse.

### Your turn

In [ ]:
# TODO 1 --- the source model, on machine A ------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  nn.CrossEntropyLoss()(model(torch.tensor(Xtr)), torch.tensor(ytr))    the training loss
#   line 2  ->  model(torch.tensor(X)).argmax(dim=1).numpy()                          predicted classes
XA_tr, yA_tr = XA[:240], yA[:240]
XA_te, yA_te = XA[240:], yA[240:]

def fit(model, Xtr, ytr, lr=0.05, epochs=600, params=None):
    """Full-batch Adam on cross entropy; `params` limits what is trained."""
    optimiser = torch.optim.Adam(model.parameters() if params is None else params, lr=lr)
    for epoch in range(epochs):
        optimiser.zero_grad()
        loss = ...                                # <- nn.CrossEntropyLoss()(model(torch.tensor(Xtr)), torch.tensor(ytr))
        loss.backward()
        optimiser.step()
    return model

def accuracy(model, X, y):
    with torch.no_grad():
        pred = ...                                # <- model(torch.tensor(X)).argmax(dim=1).numpy()
    return float((pred == y).mean())

core.set_seed(0)
source = nn.Sequential(nn.Linear(2, 16), nn.Tanh(),
                       nn.Linear(16, 16), nn.Tanh(),
                       nn.Linear(16, 3))
fit(source, XA_tr, yA_tr)
acc_source_A = accuracy(source, XA_te, yA_te)
# ------------------------------------------------------------------------------

In [ ]:
acc_source_B = accuracy(source, XB, yB)

print(f"machine A, held out : {acc_source_A:.3f}")
print(f"machine B, cold     : {acc_source_B:.3f}")
print(f"chance              : {1/3:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 4.4))
with torch.no_grad():
    pred_B = source(torch.tensor(XB)).argmax(dim=1).numpy()
core.plot_classes(XB, yB, ax=ax, predictions=pred_B,
                  title="the machine-A model, applied to machine B")
plt.show()

**What you should see.** High accuracy on machine A, and something far worse on
machine B — poor, but usually still above the 0.333 you would get by guessing.

That residual skill is the whole premise of transfer. The model has not learned
nothing about machine B; it has learned the right *shape* of the problem and
the wrong *coordinates*. Section 3 tests whether it is cheaper to correct the
coordinates than to start again.

## 3 · Three ways to spend thirty labels

Take ten labelled samples per class from machine B, and hold the rest back for
honest testing. Then three strategies on exactly the same thirty points.

| | what changes | why you might |
|---|---|---|
| **scratch** | a new network, random init | no dependence on machine A at all |
| **linear probe** | last layer only; body frozen | thirty points cannot support more |
| **fine-tune** | everything, small learning rate | the body is nearly right, not exactly |

### Your turn

In [ ]:
# TODO 2 --- scratch, linear probe, fine-tune -----------------------------------------------------
# Three `...` to replace:
#   line 1  ->  nn.Linear(16, 3)                a new, trainable head on the frozen body
#   line 2  ->  probe[-1].parameters()          train ONLY the head
#   line 3  ->  0.005                            fine-tuning learning rate, ten times lower
import copy
X_few, y_few, X_rest, y_rest = core.few_shot_split(XB, yB, per_class=10, seed=5)

core.set_seed(0)
scratch = nn.Sequential(nn.Linear(2, 16), nn.Tanh(),
                        nn.Linear(16, 16), nn.Tanh(),
                        nn.Linear(16, 3))
fit(scratch, X_few, y_few, lr=0.05)

probe = copy.deepcopy(source)
for p in probe.parameters():
    p.requires_grad = False
probe[-1] = ...                                   # <- nn.Linear(16, 3)
fit(probe, X_few, y_few, lr=0.05, params=...)     # <- probe[-1].parameters()

tuned = copy.deepcopy(source)
fit(tuned, X_few, y_few, lr=...)                  # <- 0.005

results = {"scratch":      accuracy(scratch, X_rest, y_rest),
           "linear probe": accuracy(probe,   X_rest, y_rest),
           "fine-tune":    accuracy(tuned,   X_rest, y_rest)}
# ------------------------------------------------------------------------------

In [ ]:
results["machine-A model, untouched"] = acc_source_B

print(core.error_table(
    [[name, f"{acc:.3f}"] for name, acc in results.items()],
    ["strategy", "accuracy on the 150 held-out machine-B samples"]))

fig, ax = plt.subplots(figsize=(7.4, 4.4))
names = list(results)
ax.barh(names, [results[n] for n in names],
        color=["#1f77b4", "#0f9d58", "#d94f2b", "#888888"][:len(names)])
ax.axvline(1/3, color="#111111", lw=1.2, ls="--")
ax.text(1/3, -0.6, " chance", fontsize=8, color="#111111")
ax.set_xlim(0, 1); ax.set_xlabel("held-out accuracy")
ax.set_title("Thirty labels, four outcomes")
ax.grid(alpha=0.25, axis="x")
plt.show()

**What you should see.** Both transfer strategies beat training from scratch on
thirty points, and both beat leaving the machine-A model alone. Which of the
two wins is less predictable, and that is a real finding rather than a defect
in the exercise — with a body this small, the linear probe's advantage
(fewer parameters to fit from thirty points) and fine-tuning's advantage
(the body can adapt to the rotation) are close to balanced.

Report what you measured, including the seed. Do not report the ordering as a
general law: on your own data it depends on how far the two domains sit apart
and how much of the model you can afford to move.

## 4 · The learning rate at which fine-tuning goes wrong

Fine-tuning has one classic failure. Use the learning rate you would use for a
fresh network and the first few large gradients wipe out the representation you
were trying to keep — you have paid for a pretrained model and thrown it away
before it could help.

### Your turn

In [ ]:
# TODO 3 --- where fine-tuning breaks ---------------------------------------------------------------
# Two `...` to replace, inside the loop:
#   line 1  ->  copy.deepcopy(source)             start from the pretrained model every time
#   line 2  ->  accuracy(m, X_rest, y_rest)
FT_LRS = [0.0005, 0.002, 0.005, 0.02, 0.05, 0.2]

ft_curve = []
for lr in FT_LRS:
    m = ...                                       # <- copy.deepcopy(source)
    fit(m, X_few, y_few, lr=lr)
    ft_curve.append(...)                          # <- accuracy(m, X_rest, y_rest)
# ------------------------------------------------------------------------------

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.plot(FT_LRS, ft_curve, "o-", lw=1.9, ms=6, color="#d94f2b",
        label="fine-tuned")
ax.axhline(results["scratch"], color="#1f77b4", lw=1.5, ls="--",
           label="from scratch, 30 labels")
ax.axhline(acc_source_B, color="#888888", lw=1.5, ls=":",
           label="machine-A model, untouched")
ax.set_xscale("log")
ax.set_xlabel("fine-tuning learning rate"); ax.set_ylabel("held-out accuracy")
ax.set_title("Too high, and the pretrained body stops being an asset")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

print(core.error_table([[f"{lr:g}", f"{a:.3f}"] for lr, a in zip(FT_LRS, ft_curve)],
                       ["learning rate", "held-out accuracy"]))

**What you should see.** A plateau at small learning rates, then a fall. At the
right-hand end fine-tuning converges towards the from-scratch result, which is
the tell: the model has forgotten machine A and is now learning machine B from
thirty points with extra steps.

The rule of thumb this justifies — *fine-tune at roughly a tenth of the
learning rate you would use from scratch* — is the one L11 applies to
ResNet-18 without deriving it.

## 5 · How many labels did you actually need?

Thirty was asserted. Test it.

### Your turn

In [ ]:
# TODO 4 --- the label budget --------------------------------------------------------------------
# Two `...` to replace, inside the loop:
#   line 1  ->  core.few_shot_split(XB, yB, per_class=n, seed=5)
#   line 2  ->  0.005                                              the fine-tuning learning rate
BUDGETS = [2, 5, 10, 20, 40]

curves = {"scratch": [], "fine-tune": []}
for n in BUDGETS:
    X_few_n, y_few_n, X_rest_n, y_rest_n = ...    # <- core.few_shot_split(XB, yB, per_class=n, seed=5)
    core.set_seed(0)
    m = nn.Sequential(nn.Linear(2, 16), nn.Tanh(),
                      nn.Linear(16, 16), nn.Tanh(),
                      nn.Linear(16, 3))
    fit(m, X_few_n, y_few_n, lr=0.05)
    curves["scratch"].append(accuracy(m, X_rest_n, y_rest_n))
    m = copy.deepcopy(source)
    fit(m, X_few_n, y_few_n, lr=...)              # <- 0.005
    curves["fine-tune"].append(accuracy(m, X_rest_n, y_rest_n))
# ------------------------------------------------------------------------------

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
for i, (name, curve) in enumerate(curves.items()):
    ax.plot(BUDGETS, curve, "o-", lw=1.9, ms=6,
            color=["#1f77b4", "#d94f2b"][i], label=name)
ax.axhline(acc_source_B, color="#888888", lw=1.4, ls=":",
           label="no adaptation")
ax.set_xscale("log"); ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("labelled samples per class"); ax.set_ylabel("held-out accuracy")
ax.set_title("Where the pretrained body stops paying for itself")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.** The two curves are furthest apart on the left and
converge as labels accumulate. That is the general shape and the one worth
remembering: **transfer buys you labels.** Given enough of them, starting from
scratch catches up, and the pretrained model stops being worth the complexity.

The engineering question is never "is transfer better" but "how many labels can
I afford, and which side of the crossover does that put me on".

## 6 · Save

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb03_transfer.npz")
np.savez(path,
         acc_source_A=acc_source_A, acc_source_B=acc_source_B,
         strategy_names=np.array(list(results)),
         strategy_acc=np.asarray(list(results.values()), dtype=float),
         ft_lrs=np.asarray(FT_LRS), ft_curve=np.asarray(ft_curve, dtype=float),
         budgets=np.asarray(BUDGETS),
         curve_scratch=np.asarray(curves["scratch"], dtype=float),
         curve_finetune=np.asarray(curves["fine-tune"], dtype=float))
torch.save(tuned.state_dict(), os.path.join(core.OUTPUT_DIR, "nb03_tuned.pt"))
print("wrote", path, "and nb03_tuned.pt")

## 7 · Before you move on

1. The machine-A model scored above chance on machine B before any adaptation.
   What does that tell you, and what would it have meant if it had scored
   *exactly* chance?
2. Why does fine-tuning at a high learning rate converge towards the
   from-scratch result rather than towards something worse?
3. You have thirty labels and a choice between a linear probe and full
   fine-tuning. What property of the *domain gap* should decide it?
4. In L11 the pretrained body is ResNet-18 and the new data is a few hundred
   photographs. Which of the numbers in this notebook would you expect to
   change most, and in which direction?

Next: **notebook 04**, where the fine-tuned model has to fit on a device.